# Part 3 — Feature Engineering

Build the feature matrix for the full 2023–2025 dataset.  
Key additions over Part 2: multi-year lag features, seasonal dummies, and route/carrier history computed on the full corpus.

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import config

config.assert_data_exists()

## 1. Load Raw Data

In [ ]:
df = config.load_bts_flights()
print(df.shape)

## 2. Parse Datetime Fields

In [ ]:
# YEAR, MONTH, DAY_OF_WEEK are already columns in the raw data
# Parse FL_DATE only for cyclical/ordinal features
df["fl_date"] = pd.to_datetime(df["FL_DATE"])
df["day_of_year"]  = df["fl_date"].dt.dayofyear
df["week_of_year"] = df["fl_date"].dt.isocalendar().week.astype(int)

# Cyclical encoding for month and day_of_week
df["month_sin"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["MONTH"] / 12)
df["dow_sin"]   = np.sin(2 * np.pi * df["DAY_OF_WEEK"] / 7)
df["dow_cos"]   = np.cos(2 * np.pi * df["DAY_OF_WEEK"] / 7)

## 3. Departure Time Features

In [ ]:
dep_col = "CRS_DEP_TIME"  # HHMM integer

df["dep_hour"] = df[dep_col] // 100
df["dep_min"]  = df[dep_col] % 100
df["dep_minutes_since_midnight"] = df["dep_hour"] * 60 + df["dep_min"]

df["is_peak_hour"] = df["dep_hour"].between(7, 9) | df["dep_hour"].between(16, 19)
df["is_red_eye"]   = df["dep_hour"] < 6

## 4. Route & Carrier Statistics (Historical Delay Rate)

Compute delay rate from the training split only to prevent leakage.

In [ ]:
TARGET = "ARR_DEL15"  # 1 = arrival delayed ≥15 min

# Chronological train/test split: train on 2023-2024, test on 2025
train_mask = df["YEAR"] <= 2024
df_train = df[train_mask].copy()
df_test  = df[~train_mask].copy()

print(f"Train: {len(df_train):,}  |  Test: {len(df_test):,}")

In [ ]:
df["route"] = df["ORIGIN"] + "→" + df["DEST"]

route_delay_rate   = df_train.groupby("route")[TARGET].mean().rename("route_delay_rate")
carrier_delay_rate = df_train.groupby("OP_CARRIER")[TARGET].mean().rename("carrier_delay_rate")

df = df.join(route_delay_rate, on="route").join(carrier_delay_rate, on="OP_CARRIER")

global_mean = df_train[TARGET].mean()
df["route_delay_rate"].fillna(global_mean, inplace=True)
df["carrier_delay_rate"].fillna(global_mean, inplace=True)

## 5. Flight Sequence / Tail Number Features

Tail-number propagation captures aircraft-level lateness cascades.

In [ ]:
if "TAIL_NUM" in df.columns:
    df_sorted = df.sort_values(["TAIL_NUM", "fl_date", "CRS_DEP_TIME"])
    # DEP_DELAY_NEW is departure delay capped at 0 for early departures
    df["prev_dep_delay"] = df_sorted.groupby("TAIL_NUM")["DEP_DELAY_NEW"].shift(1)
    df["prev_dep_delay"].fillna(0, inplace=True)

## 6. Select Final Feature Set

In [ ]:
FEATURE_COLS = [
    "MONTH", "month_sin", "month_cos",
    "DAY_OF_WEEK", "dow_sin", "dow_cos",
    "dep_hour", "dep_minutes_since_midnight",
    "is_peak_hour", "is_red_eye",
    "CRS_ELAPSED_TIME",
    "DISTANCE",
    "route_delay_rate",
    "carrier_delay_rate",
    # Categorical (handled natively by LightGBM, or label-encoded for XGBoost)
    "ORIGIN", "DEST", "OP_CARRIER",
]

print("Features:", len(FEATURE_COLS))
df[FEATURE_COLS + [TARGET]].head()

## 7. Save Processed Dataset

In [ ]:
out_dir = config.DATA_PART3_PROCESSED
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "flights_2023_2025_features.parquet"
df[FEATURE_COLS + [TARGET, "YEAR"]].to_parquet(out_path, index=False)
print(f"Saved: {out_path}")